### **Week 3 — Option B**

Build a portfolio chat and compare retrieval methods. Keep the write-up separate.

Start with the supplied sample content. The baseline below uses demo mode, not the live model.

**Imports**

Run once before the other cells.

In [3]:
from pathlib import Path
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import pandas as pd
import importlib.util


#### **Part 1: Get it running, then get it live**

**Set project paths**

Use the shared internship environment. The website runs with Node.js.

In [2]:
week_dir = Path.cwd()
if week_dir.name != "Week-3":
    week_dir = week_dir / "Week-3"
app_dir = week_dir / "portfolio"
results_dir = week_dir / "results"
assert (app_dir / "package.json").exists(), "open the notebook from Week-3 or the internship folder"
results_dir.mkdir(exist_ok=True)

#use node from the terminal or the installed codex runtime
node = shutil.which("node")
if node is None:
    node = str(Path.home() / ".cache/codex-runtimes/codex-primary-runtime/dependencies/node/bin/node")
assert Path(node).is_file(), "node.js was not found"
print(subprocess.check_output([node, "--version"], text=True).strip())
print(app_dir)


v24.19.0
/Users/ahthini/Desktop/LSA-internship/Week-3/portfolio


**Read the starter tests**

The untouched starter passed all 49 tests. Read the saved output without rerunning.

In [ ]:
print((results_dir / "tests_starter.txt").read_text())

✔ GET reports health without calling a model (115.14941ms)
✔ an unsupported method is refused (1.556649ms)
✔ a non-JSON body is refused (1.645726ms)
✔ an oversized context is refused (2.6269ms)
✔ no passages produces the off-topic reply, not a model call (2.573657ms)
✔ an empty question is blocked (2.236284ms)
✔ demo mode answers from the passage and labels itself (4.177284ms)
✔ the trace records retrieval, guardrails and tokens (1.645565ms)
✔ estimateTokens is four characters to a token (0.41299ms)
✔ buildMessages puts the notes in the system message (0.762689ms)
✔ demoAnswer refuses when nothing matches the question (0.676983ms)
✔ demoAnswer picks the sentence that matches the question (0.601376ms)
✔ an empty question is rejected (3.444049ms)
✔ a very long question is rejected (0.326778ms)
✔ no passages means no context (2.051691ms)
✔ a banned phrase is caught regardless of case (0.621629ms)
✔ an answer with no banned phrase passes (0.351893ms)
✔ a grounded answer passes (0.768182ms)

**Read the sample baseline**

Retrieval found 15/20 answers. Graded answers passed 11/20. These results describe the sample content in demo mode.

In [ ]:
print((results_dir / "eval_starter_demo.txt").read_text())

chunks   : chunks.json (strategy "heading", 12 chunks)
method   : keyword overlap
settings : PASSAGES=3 FLOOR=0.05 MAX_CONTEXT_CHARS=4000
key      : not set, demo mode

id  kind         retr  ans   note
------------------------------------------------------------------------------
1   answerable   ok    ok    fact present
2   answerable   ok    ok    fact present
3   answerable   ok    FAIL  fact missing from answer
4   answerable   ok    ok    fact present
5   answerable   ok    FAIL  fact missing from answer
6   answerable   ok    ok    fact present
7   answerable   ok    ok    fact present
8   answerable   ok    ok    fact present
9   answerable   ok    FAIL  fact missing from answer
10  answerable   ok    ok    fact present
11  needs_faq    MISS  FAIL  blocked: guardrail
12  needs_faq    MISS  FAIL  blocked: guardrail
13  needs_faq    MISS  FAIL  blocked: guardrail
14  must_refuse  ok    ok    refused
15  must_refuse  ok    ok    refused
16  paraphrase   MISS  FAIL  fact missing fr

**Run another local check**

Leave switched off unless needed. New checks use separate files so the original baseline is preserved.

In [ ]:
run_checks = False
if run_checks:
    env = os.environ.copy()
    env.pop("GROQ_API_KEY", None)
    checks = [("tests_latest.txt", ["--test"]),
              ("eval_latest_demo.txt", ["tests/run_eval.js"])]
    for filename, arguments in checks:
        result = subprocess.run([node, *arguments], cwd=app_dir, env=env,
                                capture_output=True, text=True)
        (results_dir / filename).write_text(result.stdout + result.stderr)
        print(filename, "exit code:", result.returncode)
        print(result.stdout + result.stderr)
else:
    print("checks switched off, original baseline preserved")

checks switched off, original baseline preserved


**Open the sample website**

Open http://localhost:3000 while the local server is running. Ask a suggested question and expand **What happened**. The answer should say **DEMO MODE**.

The terminal command to start it again is in README.md. Stop the server with Ctrl+C in that terminal.

**Next: GitHub and deployment**

Commit the untouched starter, then deploy it on Vercel with `Week-3/portfolio` as the project root. Verify the live app before changing the sample content.

After deployment, personalise the notes and evaluation questions, then begin the three chunking comparisons.

#### **Task 5: Chunk three ways**

**Choose the metric**

Compare whole files, headings and overlapping windows. Prioritise graded accuracy, then context waste and characters sent. Keep the notes, questions, prompt and settings fixed. Demo results measure the extractive fallback, not Groq.


In [3]:
#save each strategy separately, leaving the website chunks unchanged
chunk_dir = results_dir / "task5_chunks"
chunk_dir.mkdir(exist_ok=True)
strategies = ["whole", "heading", "window"]
chunk_rows = []
for strategy in strategies:
    destination = chunk_dir / f"{strategy}.json"
    result = subprocess.run([sys.executable, "prep/prep.py", strategy, "--with-faq",
                             "-o", str(destination.resolve())], cwd=app_dir,
                            capture_output=True, text=True, check=True)
    doc = json.loads(destination.read_text())
    sizes = [chunk["chars"] for chunk in doc["chunks"]]
    chunk_rows.append({"strategy": strategy, "chunks": len(sizes),
                       "min_chars": min(sizes), "mean_chars": round(sum(sizes) / len(sizes)),
                       "max_chars": max(sizes)})
chunk_shapes = pd.DataFrame(chunk_rows)
display(chunk_shapes)

,strategy,chunks,min_chars,mean_chars,max_chars
0,whole,5,1061,1641,3101
1,heading,23,130,352,609
2,window,19,152,540,600


**Run the comparison**

Run in demo mode first. For Groq, set `use_live_model = True` after saving the key in `portfolio/.env.local`. Live mode makes up to 60 API calls. Saved runs are reused. Set `rerun_evaluation = True` only for a deliberate repeat.


In [4]:
use_live_model = True
rerun_evaluation = False
mode = "groq" if use_live_model else "demo"
run_dir = results_dir / f"task5_{mode}_paced"
run_dir.mkdir(exist_ok=True)
env = os.environ.copy()
env.pop("GROQ_API_KEY", None)
node_options = []
if use_live_model:
    assert (app_dir / ".env.local").exists(), "save GROQ_API_KEY in portfolio/.env.local first"
    node_options = ["--env-file=.env.local"]

#fingerprint the experiment so changed inputs cannot reuse old scores
inputs = [app_dir / "questions.json", app_dir / "config.js", app_dir / "tests/run_eval.js",
          app_dir / "api/chat.js", app_dir / "lib/retrieve.js", app_dir / "lib/guard.js",
          *sorted((app_dir / "content").glob("*.md"))]
fingerprint = hashlib.sha256(b"".join(path.read_bytes() for path in inputs)).hexdigest()
manifest_path = run_dir / "manifest.json"
if manifest_path.exists() and not rerun_evaluation:
    assert json.loads(manifest_path.read_text())["fingerprint"] == fingerprint, \
        "inputs changed: archive the old results, then set rerun_evaluation = True"
else:
    manifest_path.write_text(json.dumps({"mode": mode, "fingerprint": fingerprint,
                                       "primary_metric": "graded_accuracy"}, indent=2))

for strategy in strategies:
    log_path = run_dir / f"{strategy}.txt"
    if log_path.exists() and not rerun_evaluation:
        print(f"{strategy}: using saved results")
        continue
    result = subprocess.run([node, *node_options, "tests/run_eval.js", "--chunks",
                             str((chunk_dir / f"{strategy}.json").resolve()),
                             "--details", str((run_dir / f"{strategy}_details.json").resolve())], cwd=app_dir,
                            env=env, capture_output=True, text=True)
    output = result.stdout + result.stderr
    if result.returncode != 0:
        (run_dir / f"{strategy}_incomplete.txt").write_text(output)
    assert result.returncode == 0, f"{strategy} failed: {output[-1500:]}"
    if use_live_model:
        assert "set, live model" in output, "no key loaded: this was a demo run, not a live score"
    log_path.write_text(output)
    print(f"{strategy}: saved to {log_path.name}")

whole: saved to whole.txt
heading: saved to heading.txt
window: saved to window.txt


**Compare the results**

MRR measures ranking. Waste measures irrelevant retrieved text. Graded accuracy checks expected answer phrases, so inspect failures too. Do not compare these scores directly with the sample baseline because the notes and questions changed.


In [5]:
comparison_rows = []
for strategy in strategies:
    output = (run_dir / f"{strategy}.txt").read_text()
    def metric(pattern):
        match = re.search(pattern, output)
        assert match, f"missing metric for {strategy}"
        return float(match.group(1))
    comparison_rows.append({"strategy": strategy, "mode": mode,
        "chunks": len(json.loads((chunk_dir / f"{strategy}.json").read_text())["chunks"]),
        "retrieval_percent": metric(r"retrieval found the fact.*\(([\d.]+)%\)"),
        "mrr": metric(r"retrieval MRR\s*:\s*([\d.]+)"),
        "waste_percent": metric(r"mean context waste\s*:\s*([\d.]+)"),
        "mean_chars": int(metric(r"mean chars per question\s*:\s*([\d.]+)")),
        "graded_percent": metric(r"graded correct.*\(([\d.]+)%\)")})
comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(run_dir / "comparison.csv", index=False)
display(comparison)

,strategy,mode,chunks,retrieval_percent,mrr,waste_percent,mean_chars,graded_percent
0,whole,groq,5,95.0,0.867,28.3,2276,70.0
1,heading,groq,23,80.0,0.775,46.3,920,70.0
2,window,groq,19,85.0,0.742,46.7,1456,75.0


**Inspect the failures**

Read which questions failed for each strategy. Use the trace on the website to distinguish missing passages from poor answers. Keep the test questions unchanged.


In [6]:
for strategy in strategies:
    output = (run_dir / f"{strategy}.txt").read_text()
    print(f"\n{strategy} ({mode})")
    failures = [line for line in output.splitlines() if "FAIL" in line or "MISS" in line]
    print("\n".join(failures) if failures else "all questions passed the automatic checks")


whole (groq)
3   answerable   ok    FAIL  fact missing from answer
6   answerable   ok    FAIL  fact missing from answer
7   answerable   ok    FAIL  fact missing from answer
16  paraphrase   MISS  FAIL  blocked: guardrail
18  paraphrase   ok    FAIL  fact missing from answer
19  paraphrase   ok    FAIL  fact missing from answer

heading (groq)
3   answerable   ok    FAIL  fact missing from answer
7   answerable   ok    FAIL  fact missing from answer
16  paraphrase   MISS  FAIL  blocked: guardrail
17  paraphrase   MISS  FAIL  blocked: grounded
18  paraphrase   MISS  FAIL  fact missing from answer
20  paraphrase   MISS  FAIL  blocked: grounded

window (groq)
3   answerable   ok    FAIL  fact missing from answer
7   answerable   ok    FAIL  fact missing from answer
16  paraphrase   MISS  FAIL  blocked: guardrail
18  paraphrase   MISS  FAIL  blocked: grounded
20  paraphrase   MISS  FAIL  blocked: grounded


**Rescore typography differences**

Normalise hyphens and whitespace in saved answers. Only question 3 changes. Original scores stay above for comparison. No API calls.


In [ ]:
#rescore saved answers without calling groq
result = subprocess.run([node, "tests/rescore_saved.js"], cwd=app_dir,
                        capture_output=True, text=True, check=True)
rescored = pd.DataFrame(json.loads((results_dir / "task5_groq_paced" / "rescored_comparison.json").read_text()))
display(rescored)


[
  {
    "strategy": "whole",
    "correct": 15,
    "items": 20,
    "graded_percent": 75,
    "changed_ids": "3"
  },
  {
    "strategy": "heading",
    "correct": 15,
    "items": 20,
    "graded_percent": 75,
    "changed_ids": "3"
  },
  {
    "strategy": "window",
    "correct": 16,
    "items": 20,
    "graded_percent": 80,
    "changed_ids": "3"
  }
]

#### **Task 6: TF-IDF retrieval**

**Build both variants**

Use the same 19 window chunks. Compare English stopword removal with keeping every word. Python and JavaScript use the same token pattern.


In [3]:
task6_dir = results_dir / "task6"
task6_dir.mkdir(exist_ok=True)
spec = importlib.util.spec_from_file_location("vectorise", app_dir / "prep/vectorise.py")
vector_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(vector_module)
window_doc = json.loads((results_dir / "task5_chunks/window.json").read_text())
variants = {"stopwords_removed": "english", "stopwords_kept": None}
vector_rows = []
for name, stop_words in variants.items():
    doc, vectoriser, matrix = vector_module.vectorise(window_doc, stop_words=stop_words)
    (task6_dir / f"{name}.json").write_text(json.dumps(doc, indent=1, ensure_ascii=False))
    vector_rows.append({"variant": name, "chunks": len(doc["chunks"]),
                        "vocabulary": doc["vectoriser"]["vocab_size"]})
display(pd.DataFrame(vector_rows))


,variant,chunks,vocabulary
0,stopwords_removed,19,416
1,stopwords_kept,19,470


**Compare retrieval and demo answers**

No API calls. Compare keyword matching and both TF-IDF variants on all 20 questions. Refusals are scored separately. These demo answer scores are not comparable with the previous Groq answer scores.


In [4]:
task6_rows = []
for name in ["keyword", *variants]:
    chunk_file = results_dir / "task5_chunks/window.json" if name == "keyword" else task6_dir / f"{name}.json"
    vector_flag = [] if name == "keyword" else ["--vector"]
    env = os.environ.copy()
    env.pop("GROQ_API_KEY", None)
    result = subprocess.run([node, "tests/run_eval.js", "--chunks", str(chunk_file.resolve()),
                             *vector_flag, "--details", str((task6_dir / f"{name}_details.json").resolve())],
                            cwd=app_dir, env=env, capture_output=True, text=True, check=True)
    output = result.stdout
    (task6_dir / f"{name}_demo.txt").write_text(output)
    details = json.loads((task6_dir / f"{name}_details.json").read_text())
    def read_metric(pattern):
        return float(re.search(pattern, output).group(1))
    task6_rows.append({"method": name, "mode": "demo",
        "retrieval_percent": read_metric(r"retrieval found the fact.*\(([\d.]+)%\)"),
        "mrr": read_metric(r"retrieval MRR\s*:\s*([\d.]+)"),
        "waste_percent": read_metric(r"mean context waste\s*:\s*([\d.]+)"),
        "mean_chars": int(read_metric(r"mean chars per question\s*:\s*([\d.]+)")),
        "graded_percent": read_metric(r"graded correct.*\(([\d.]+)%\)"),
        "refusals_correct": sum(r["grade"]["correct"] for r in details if r["kind"] == "must_refuse"),
        "paraphrases_correct": sum(r["grade"]["correct"] for r in details if r["kind"] == "paraphrase")})
task6_comparison = pd.DataFrame(task6_rows)
task6_comparison.to_csv(task6_dir / "comparison_demo.csv", index=False)
display(task6_comparison)


,method,mode,retrieval_percent,mrr,waste_percent,mean_chars,graded_percent,refusals_correct,paraphrases_correct
0,keyword,demo,85.0,0.742,46.7,1456,60.0,2,1
1,stopwords_removed,demo,85.0,0.792,48.7,1433,60.0,2,1
2,stopwords_kept,demo,85.0,0.742,45.1,1696,50.0,0,1


**Use TF-IDF on the website**

Select the variant with stopwords removed. The app and scoring page both load its vectors. This updates local files, publishing requires a commit and push.


In [5]:
shutil.copyfile(task6_dir / "stopwords_removed.json", app_dir / "chunks.json")
print("website chunks updated: window chunks with TF-IDF and English stopwords removed")


website chunks updated: window chunks with TF-IDF and English stopwords removed


#### **Task 7: Show your sources**

The website now shows source headings, chunk IDs and similarity scores below each answer. Only passages that fit in the context are listed. Similarity is not answer confidence.


#### **Task 8: Find the ceiling**

**Inspect the paraphrases**

Compare the same five paraphrases across keyword and TF-IDF runs. These are demo answers. A missing fact cannot be recovered by rewriting the generation prompt.


In [32]:
paraphrase_rows = []
for method in ["keyword", "stopwords_removed", "stopwords_kept"]:
    details = json.loads((task6_dir / f"{method}_details.json").read_text())
    for row in details:
        if row["kind"] == "paraphrase":
            paraphrase_rows.append({"method": method, "id": row["id"],
                "question": row["question"], "passages": ", ".join(row["passages"]),
                "correct": row["grade"]["correct"], "reason": row["grade"]["why"],
                "answer": row.get("answer", "")})
paraphrase_comparison = pd.DataFrame(paraphrase_rows)
paraphrase_comparison.to_csv(task6_dir / "paraphrase_comparison.csv", index=False)
display(paraphrase_comparison[["method", "id", "correct", "reason"]])


,method,id,correct,reason
0,keyword,16,False,blocked: guardrail
1,keyword,17,True,fact present
2,keyword,18,False,fact missing from answer
3,keyword,19,False,fact missing from answer
4,keyword,20,False,fact missing from answer
5,stopwords_removed,16,False,blocked: guardrail
6,stopwords_removed,17,True,fact present
7,stopwords_removed,18,False,fact missing from answer
8,stopwords_removed,19,False,fact missing from answer
9,stopwords_removed,20,False,fact missing from answer


#### **Task 9: Dense embeddings offline**

**Attempt semantic retrieval**

Use the same window chunks and 20 questions. The model needs sentence-transformers and a model download. Record an error if unavailable, as permitted by the brief. Keep TF-IDF on the website. The 0.15 dense threshold follows the supplied example and is not equivalent to the lexical threshold.


In [34]:
try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    dense_doc = json.loads((results_dir / "task5_chunks/window.json").read_text())
    questions = json.loads((app_dir / "questions.json").read_text())["questions"]
    matrix = model.encode([c["text"] for c in dense_doc["chunks"]], normalize_embeddings=True)
    query_vectors = model.encode([q["q"] for q in questions], normalize_embeddings=True)
    dense_rows = []
    for item, query in zip(questions, query_vectors):
        scores = matrix @ query
        order = sorted(range(len(scores)), key=lambda i: -float(scores[i]))[:3]
        hits = [dense_doc["chunks"][i] for i in order if scores[i] > 0.15]
        if item["kind"] == "must_refuse":
            found = not hits
            rr = float(found)
        else:
            rank = next((i for i, chunk in enumerate(hits, 1)
                         if any(e.lower() in chunk["text"].lower() for e in item["expect_any"])), None)
            found = rank is not None
            rr = 1 / rank if rank else 0
        dense_rows.append({"id": item["id"], "kind": item["kind"], "found": found, "rr": rr})
    dense_results = pd.DataFrame(dense_rows)
    dense_results.to_csv(task6_dir / "dense_retrieval.csv", index=False)
    dense_status = {"status": "completed", "retrieval_percent": float(dense_results["found"].mean() * 100),
                    "mrr": float(dense_results["rr"].mean()), "note": "retrieval only, no generated-answer score"}
except Exception as error:
    dense_status = {"status": "not completed", "error": f"{type(error).__name__}: {error}"}
(task6_dir / "dense_status.json").write_text(json.dumps(dense_status, indent=2))
print(json.dumps(dense_status, indent=2))


{
  "status": "not completed",
  "error": "ModuleNotFoundError: No module named 'sentence_transformers'"
}


**Live TF-IDF results**

Both variants completed with zero API errors. Stopwords removed scored 75%, kept scored 90%. Both refused 2/2, but keeping stopwords retrieved irrelevant passages and relied on the grounding guardrail. Keyword window scored 80% in the earlier normalised Groq run. These are single runs, not evidence that one method always wins.


In [1]:
#read saved live runs without making api calls
live_dir = task6_dir / "live"
live_comparison = pd.DataFrame(json.loads((live_dir / "comparison.json").read_text()))
display(live_comparison)


,method,mode,graded_percent,refusals_correct,paraphrases_correct,api_errors
0,stopwords_removed,groq,75.0,2,1,0
1,stopwords_kept,groq,90.0,2,4,0
